In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

BASE = Path.cwd()
VAL_DIR = BASE / 'ci_recover_validation_2026-11-17'
CASE_FILE = VAL_DIR / 'ci_recover_78_with_validation.csv'
PICK_FILE = VAL_DIR / 'validation_per_pick.csv'
TEST_FILE = VAL_DIR / 'validation_per_test.csv'

cases = pd.read_csv(CASE_FILE)
picks = pd.read_csv(PICK_FILE)
tests = pd.read_csv(TEST_FILE)
print('cases:', cases.shape, 'picks:', picks.shape, 'tests:', tests.shape)

In [ ]:
# The paper groups all non-eligible outcomes as follows:
# Invalid = deleted test + skipped + flaky; Other = all NOT_TESTABLE_*;
# Not evaluated = NEEDS_2_GPUS.
def paper_outcome(verdict):
    if verdict in {'VALIDATED', 'FIX_DOES_NOT_FIX', 'NOT_REPRODUCED'}: return 'Eligible'
    if verdict in {'INVALID_TEST_DELETED_BY_FIX', 'SKIPPED_AT_BOTH', 'FLAKY'}: return 'Invalid'
    if verdict == 'NEEDS_2_GPUS': return 'Not evaluated'
    if str(verdict).startswith('NOT_TESTABLE_'): return 'Other'
    return 'Unmapped'

cases['paper_outcome'] = cases['val_verdict'].map(paper_outcome)
arch_order = ['gfx950', 'gfx942', 'gfx90a']
outcome_order = ['Candidates', 'Eligible', 'Validated', 'Persistent Failure', 'Persistent Pass', 'Invalid', 'Not evaluated', 'Other']
# Detailed eligible outcomes are retained for auditability.
print(cases.groupby(['val_ci_arch', 'val_verdict']).size().unstack(fill_value=0).reindex(arch_order))

table1 = pd.DataFrame(index=arch_order)
table1['Candidates'] = cases.groupby('val_ci_arch').size().reindex(arch_order, fill_value=0)
table1['Eligible'] = cases[cases['paper_outcome'] == 'Eligible'].groupby('val_ci_arch').size().reindex(arch_order, fill_value=0)
table1['Validated'] = cases.query("val_verdict == 'VALIDATED'").groupby('val_ci_arch').size().reindex(arch_order, fill_value=0)
table1['Persistent Failure'] = cases.query("val_verdict == 'FIX_DOES_NOT_FIX'").groupby('val_ci_arch').size().reindex(arch_order, fill_value=0)
table1['Persistent Pass'] = cases.query("val_verdict == 'NOT_REPRODUCED'").groupby('val_ci_arch').size().reindex(arch_order, fill_value=0)
table1 = table1.reindex(columns=outcome_order, fill_value=0).astype(int)
table1.loc['Total'] = table1.sum(numeric_only=True)
table1

In [ ]:
eligible = cases[cases['paper_outcome'] == 'Eligible']
validated = cases[cases['val_verdict'] == 'VALIDATED']
print(f"Eligible: {len(eligible)}; validated: {len(validated)}; rate: {len(validated)/len(eligible):.1%}")
print('Validated repair cases:', len(validated))
print('Unique validated repair commits:', validated['val_best_pick'].nunique())
print('Validated commits by kind:', validated.groupby('val_best_pick_kind')['val_best_pick'].nunique().to_dict())

In [ ]:
def job_family(value):
    value = str(value).lower()
    if 'distributed' in value: return 'Distributed'
    if 'default' in value: return 'Default'
    return 'Other'

validated = validated.copy()
validated['job_family'] = validated['job_name_actions'].map(job_family)
print('Workflow families:')
print(validated['workflow_name_actions'].value_counts())
print('Job families:')
print(validated['job_family'].value_counts())
print('Assertion clusters:')
print(validated['assertion_error_cluster'].value_counts())

In [ ]:
# The paper defines coverage over runs that returned a strong/weak patch match.
# `run*_present` is broader: it also marks no_match and indirect_recovery rows.
run_names = ['run2', 'run3', 'run4']
category_cols = [f'{r}_match_category' for r in run_names]
presence = cases[category_cols].fillna('').astype(str).apply(
    lambda col: col.str.lower().str.contains('strong_patch_match|weak_patch_match', regex=True)
)
coverage = presence.sum(axis=1).value_counts().sort_index()
print('Cases covered by exactly 1/2/3 runs:', coverage.to_dict())

def agreement(sub, columns, n):
    vals = sub[columns].astype('string').apply(lambda col: col.str.lower().str.strip())
    vals = vals.replace({'none': pd.NA, 'nan': pd.NA, '': pd.NA})
    vals = vals.where(presence.loc[sub.index].to_numpy())
    resolved = vals.notna().sum(axis=1).eq(n)
    same = vals.nunique(axis=1).eq(1)
    return int((resolved & same).sum()), int(resolved.sum())

for n in [2, 3]:
    sub = cases.loc[presence.sum(axis=1).eq(n)]
    print(f'\n{n}-run cases:', len(sub))
    for label, cols in {
        'category': [f'{r}_match_category.1' for r in run_names],
        'commit': [f'{r}_related_commit_sha' for r in run_names],
    }.items():
        print(label, agreement(sub, cols, n))

# Joint agreement requires the same category and the same candidate commit.
def joint_agreement(sub, n):
    cat = sub[[f'{r}_match_category.1' for r in run_names]].astype('string').apply(lambda col: col.str.lower().str.strip())
    sha = sub[[f'{r}_related_commit_sha' for r in run_names]].astype('string').apply(lambda col: col.str.lower().str.strip())
    cat = cat.where(presence.loc[sub.index].to_numpy())
    sha = sha.replace({'none': pd.NA, 'nan': pd.NA, '': pd.NA}).where(presence.loc[sub.index].to_numpy())
    values = []
    for idx in sub.index:
        pairs = [(cat.loc[idx, f'{r}_match_category.1'], sha.loc[idx, f'{r}_related_commit_sha']) for r in run_names if pd.notna(cat.loc[idx, f'{r}_match_category.1']) and pd.notna(sha.loc[idx, f'{r}_related_commit_sha'])]
        values.append(len(pairs) == n and len(set(pairs)) == 1)
    return int(sum(values)), len(values)

for n in [2, 3]:
    sub = cases.loc[presence.sum(axis=1).eq(n)]
    print(f'{n}-run joint category+commit agreement:', joint_agreement(sub, n))

# Validation success grouped by the number of runs assigning strong_patch_match.
eligible_for_linkage = cases[cases['paper_outcome'] == 'Eligible'].copy()
strong_cols = [f'{r}_match_category.1' for r in run_names]
eligible_for_linkage['strong_match_runs'] = eligible_for_linkage[strong_cols].apply(
    lambda row: sum(str(value).strip().lower() == 'strong_patch_match' for value in row if pd.notna(value)), axis=1
)
strong_validation_summary = eligible_for_linkage.groupby('strong_match_runs').agg(
    cases=('val_verdict', 'size'),
    validated=('val_verdict', lambda s: (s == 'VALIDATED').sum()),
)
strong_validation_summary['validation_rate'] = strong_validation_summary['validated'] / strong_validation_summary['cases']
strong_validation_summary.reindex([0, 1, 2, 3], fill_value=0)

In [ ]:
print('Per-pick verdicts:')
print(picks['verdict'].value_counts())
print('Per-test classes:')
print(tests['class'].value_counts())

# Independent check of the case-level validation rule.
test_summary = tests.groupby(['case_key', 'agent_pick'])['class'].agg(list).reset_index()
def test_rule(classes):
    if 'F2P' in classes and 'P2F' not in classes: return 'VALIDATED'
    if any(c == 'F2F' for c in classes): return 'FIX_DOES_NOT_FIX'
    if all(c == 'P2P' for c in classes): return 'NOT_REPRODUCED'
    if 'FLAKY' in classes: return 'FLAKY'
    return 'OTHER'
test_summary['recomputed_rule'] = test_summary['class'].map(test_rule)
test_summary['recomputed_rule'].value_counts()

In [ ]:
import os
import requests
from urllib.parse import quote

GITHUB_REPO = 'pytorch/pytorch'
GITHUB_API_URL = 'https://api.github.com'
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '')
github_headers = {'Accept': 'application/vnd.github+json'}
if GITHUB_TOKEN:
    github_headers['Authorization'] = f'Bearer {GITHUB_TOKEN}'

compare_cache = {}
def get_window_stats(start_sha, end_sha):
    start, end = str(start_sha).strip(), str(end_sha).strip()
    if not start or not end or start.lower() in {'nan', 'none'} or end.lower() in {'nan', 'none'}:
        return {'window_commit_distance': pd.NA, 'window_compare_status': 'missing_commit'}
    key = (start, end)
    if key in compare_cache: return compare_cache[key]
    url = f'{GITHUB_API_URL}/repos/{GITHUB_REPO}/compare/{quote(start, safe=chr(0))}...{quote(end, safe=chr(0))}'
    try:
        response = requests.get(url, headers=github_headers, timeout=30)
        payload = response.json()
        response.raise_for_status()
        result = {'window_commit_distance': payload.get('total_commits'), 'window_compare_status': payload.get('status', 'unknown')}
    except Exception as exc:
        result = {'window_commit_distance': pd.NA, 'window_compare_status': f"http_{getattr(response, 'status_code', 'error')}", 'window_compare_error': str(exc)}
    compare_cache[key] = result
    return result

window_stats = pd.DataFrame([get_window_stats(r.window_start_commit, r.window_end_commit) for r in validated.itertuples()])
window_df = pd.concat([validated.reset_index(drop=True), window_stats], axis=1)
window_values = pd.to_numeric(window_df['window_commit_distance'], errors='coerce').dropna()
window_summary = pd.Series({'count': len(window_values), 'average': window_values.mean(), 'median': window_values.median(), 'minimum': window_values.min(), 'maximum': window_values.max(), 'range': window_values.max() - window_values.min()})
window_summary

In [ ]:
commit_cache = {}
def get_commit_stats(sha):
    sha = str(sha).strip()
    if not sha or sha.lower() in {'nan', 'none'}: return {'github_api_status': 'missing_sha'}
    if sha in commit_cache: return commit_cache[sha]
    url = f'{GITHUB_API_URL}/repos/{GITHUB_REPO}/commits/{quote(sha, safe=chr(0))}'
    try:
        response = requests.get(url, headers=github_headers, params={'per_page': 100}, timeout=30)
        payload = response.json()
        response.raise_for_status()
        files = list(payload.get('files', []))
        next_url = response.links.get('next', {}).get('url')
        while next_url:
            page = requests.get(next_url, headers=github_headers, timeout=30)
            page_payload = page.json()
            page.raise_for_status()
            files.extend(page_payload.get('files', []))
            next_url = page.links.get('next', {}).get('url')
        stats = payload.get('stats') or {}
        result = {'github_commit_sha': payload.get('sha', sha), 'changed_files': len(files), 'additions': stats.get('additions'), 'deletions': stats.get('deletions'), 'github_api_status': 'ok'}
    except Exception as exc:
        result = {'github_commit_sha': sha, 'changed_files': pd.NA, 'additions': pd.NA, 'deletions': pd.NA, 'github_api_status': f"http_{getattr(response, 'status_code', 'error')}", 'github_api_error': str(exc)}
    commit_cache[sha] = result
    return result

pick_shas = validated['val_best_pick'].dropna().astype(str).unique()
pick_stats = pd.DataFrame([get_commit_stats(sha) for sha in pick_shas])
validated_pick_stats = validated.merge(pick_stats, left_on='val_best_pick', right_on='github_commit_sha', how='left', validate='many_to_one')
print('validated cases:', len(validated), 'unique picks queried:', len(pick_shas), 'API successes:', (pick_stats['github_api_status'] == 'ok').sum())
validated_pick_stats[['changed_files', 'additions', 'deletions']].describe().T